# exploracao_gold_receita

Testes e conferência das tabelas do `gold_receita`: `gold.agg_receita_mensal` e `gold.agg_preco_volume`.
**Não entra no Job:** só lê e exibe. Rode depois do `gold_receita`. O que esperar está na última célula.

1. Estrutura: linhas, schema, grão e nulos
2. Reconciliação mês a mês com a Silver
3. Métricas do case (P1 × P2)
4. Preço × volume fecha
5. Recortes: série mensal, classe, município e onde está a queda
6. Sanidade: segmentos de um período só, kWh negativo e tarifa fora do padrão
7. Nomes de município e bairro: lista completa, repetidos, deformados e parecidos
8. Nomes no cadastro (Silver): titulares repetidos e deformados

In [0]:
%run ../bronze-to-silver/funcoes_comuns

In [0]:
CATALOGO = "workspace"
SEGMENTO = ["municipio", "bairro", "classe"]
P1, P2 = ("2022-07", "2023-06"), ("2025-07", "2026-06")

rm = spark.table(f"{CATALOGO}.gold.agg_receita_mensal")
pv = spark.table(f"{CATALOGO}.gold.agg_preco_volume")
for nome, df in [("agg_receita_mensal", rm), ("agg_preco_volume", pv)]:
    print(f"{nome}: {df.count():,} linhas")
    df.printSchema()

In [0]:
# 1. Grão, nulos e domínios (falham com mensagem clara se algo estiver errado)
checar_unicidade(rm, ["mes_referencia", *SEGMENTO], "agg_receita_mensal")
checar_sem_nulos(rm, ["mes_referencia", *SEGMENTO, "receita_total", "receita_energia", "tributos", "faturas"], "agg_receita_mensal")
checar_dominio(rm, "classe", ["RESIDENCIAL", "COMERCIAL", "INDUSTRIAL", "RURAL", "PODER PUBLICO"], "agg_receita_mensal")
checar_unicidade(pv, SEGMENTO, "agg_preco_volume")
checar_sem_nulos(pv, [*SEGMENTO, "variacao_receita", "efeito_volume", "efeito_preco"], "agg_preco_volume")
print("meses:", rm.select("mes_referencia").distinct().count(), "(esperado: 48)")

In [0]:
# 2. Reconciliação mês a mês com a Silver: a lista de diferenças tem que vir vazia
ft = spark.table(f"{CATALOGO}.silver.tb_faturas")
lt = spark.table(f"{CATALOGO}.silver.tb_leituras").join(ft.select("id_uc", "mes_referencia"), ["id_uc", "mes_referencia"])
silver = (ft.groupBy("mes_referencia").agg(F.sum("valor_total").alias("receita_silver"), F.count("*").alias("faturas_silver"))
            .join(lt.groupBy("mes_referencia").agg(F.sum("consumo_kwh").alias("kwh_silver")), "mes_referencia", "full"))
gold = rm.groupBy("mes_referencia").agg(F.sum("receita_total").alias("receita_gold"), F.sum("faturas").alias("faturas_gold"),
                                        F.sum("consumo_kwh").alias("kwh_gold"))
dif = gold.join(silver, "mes_referencia", "full").filter(
    ~F.col("receita_gold").eqNullSafe(F.col("receita_silver")) | ~F.col("faturas_gold").eqNullSafe(F.col("faturas_silver"))
    | ~F.col("kwh_gold").eqNullSafe(F.col("kwh_silver")))
print(f"meses com diferença entre Gold e Silver: {dif.count()} (esperado: 0)")
display(dif.orderBy("mes_referencia"))

In [0]:
# 3. Métricas do case: receita +8% e volume -4% entre P1 e P2
periodo = F.when(F.col("mes_referencia").between(*P1), "P1").when(F.col("mes_referencia").between(*P2), "P2")
case = (rm.withColumn("periodo", periodo).filter(F.col("periodo").isNotNull()).groupBy("periodo")
          .agg(F.sum("receita_total").alias("receita_total"), F.sum("receita_energia").alias("receita_energia"),
               F.sum("consumo_kwh").alias("kwh"))
          .withColumn("tarifa_media", F.try_divide("receita_energia", "kwh")).orderBy("periodo"))
display(case)
p1, p2 = case.collect()
for c in ["receita_total", "receita_energia", "kwh", "tarifa_media"]:
    print(f"{c}: {(p2[c] / p1[c] - 1) * 100:+.1f}%")

In [0]:
# 4. Preço × volume: em toda linha e no total, volume + preço = variação da receita de energia
print("linhas que não fecham:", pv.filter(F.col("efeito_volume") + F.col("efeito_preco") != F.col("variacao_receita")).count(),
      "(esperado: 0)")
display(pv.agg(*[F.sum(c).alias(c) for c in ["receita_energia_p1", "receita_energia_p2", "variacao_receita",
                                            "efeito_volume", "efeito_preco"]]))

In [0]:
# 5a. Série mensal (use o gráfico de linha do display)
display(rm.groupBy("mes_referencia")
          .agg(F.sum("receita_total").alias("receita_total"), F.sum("consumo_kwh").alias("kwh"),
               F.try_divide(F.sum("receita_energia"), F.sum("consumo_kwh")).alias("tarifa_media"))
          .orderBy("mes_referencia"))

In [0]:
# 5b. Por classe e por município: variação de kWh, receita e tarifa entre P1 e P2
def por(*cols):
    a = pv.groupBy(*cols).agg(*[F.sum(c).alias(c) for c in ["kwh_p1", "kwh_p2", "receita_energia_p1",
                                                          "receita_energia_p2", "efeito_volume", "efeito_preco"]])
    pct = lambda novo, velho: F.round((F.try_divide(novo, velho) - 1) * 100, 1)   # nulo se a base for 0
    tarifa = lambda p: F.try_divide(f"receita_energia_{p}", f"kwh_{p}")
    return (a.withColumns({"kwh_%": pct(F.col("kwh_p2"), F.col("kwh_p1")),
                           "receita_%": pct(F.col("receita_energia_p2"), F.col("receita_energia_p1")),
                           "tarifa_%": pct(tarifa("p2"), tarifa("p1"))})
             .orderBy("efeito_volume"))

display(por("classe"))      # o case fala em reajuste residencial de 10,5%: compare com tarifa_%
display(por("municipio"))

In [0]:
# 5c. Onde está a queda: segmentos com maior perda de volume
display(pv.withColumn("kwh_%", F.round((F.try_divide("kwh_p2", "kwh_p1") - 1) * 100, 1))
          .orderBy("efeito_volume").limit(20))

In [0]:
# 6. Sanidade
print("segmentos só com P1 ou só com P2:", pv.filter((F.col("kwh_p1") == 0) | (F.col("kwh_p2") == 0)).count())
print("linhas mensais com kWh negativo:", rm.filter(F.col("consumo_kwh") < 0).count())
print("linhas mensais sem kWh (tarifa nula):", rm.filter(F.col("tarifa_media").isNull()).count())
display(rm.groupBy("classe").agg(F.min("tarifa_media").alias("min"), F.expr("percentile(tarifa_media, 0.5)").alias("mediana"),
                                 F.max("tarifa_media").alias("max")))
perfilar_coluna(rm, "tarifa_media", n=5)

In [0]:
# 7a. Lista completa de município × bairro (tabela grande: use a busca e o filtro do display)
nomes = (rm.groupBy("municipio", "bairro")
           .agg(F.countDistinct("mes_referencia").alias("meses"), F.countDistinct("classe").alias("classes"),
                F.sum("faturas").alias("faturas"), F.sum("receita_total").alias("receita_total")))
print(f"municípios: {nomes.select('municipio').distinct().count()} | pares município × bairro: {nomes.count()}")
display(nomes.orderBy("municipio", "bairro").limit(1000))

In [0]:
# 7b. Mesmo nome escrito de jeitos diferentes: depois de tirar acento, pontuação e espaço extra, a chave tem que ser única
chave = lambda c: F.regexp_replace(padronizar_texto(c), r"[^A-Z0-9 ]", "")
for col, grupo in [("municipio", []), ("bairro", ["municipio"])]:
    rep = (nomes.groupBy(*grupo, chave(col).alias("chave"))
                .agg(F.countDistinct(col).alias("grafias"), F.collect_set(col).alias("como_aparece"))
                .filter("grafias > 1"))
    print(f"{col}: {rep.count()} nomes com mais de uma grafia (esperado: 0)")
    display(rep)

In [0]:
# 7c. Nomes deformados: encoding quebrado (ex.: "JORDÃ£O"), caractere estranho, espaço nas pontas ou duplo, nome curto
deformado = {
    "encoding_quebrado": r"[ÃÂ][\x{80}-\x{BF}]",
    "caractere_estranho": r"[^\p{L}\p{N} .,'()/ºª-]",
    "espaco_extra": r"^\s|\s$|\s{2,}",
    "nome_curto": r"^.{1,3}$",
}
for col in ["municipio", "bairro"]:
    for problema, regex in deformado.items():
        achados = nomes.select(col).distinct().filter(F.col(col).rlike(regex))
        n = achados.count()
        print(f"{col} · {problema}: {n} (esperado: 0)")
        if n:
            display(achados)

In [0]:
# 7d. Nomes parecidos no mesmo município (até 2 letras de diferença, sem acento). Revisão manual:
# "CAETÉS I"/"CAETÉS II" são bairros diferentes de verdade; um erro de digitação apareceria aqui
b = nomes.select("municipio", "bairro").distinct()
parecidos = (b.alias("a").join(b.alias("b"), (F.col("a.municipio") == F.col("b.municipio")) & (F.col("a.bairro") < F.col("b.bairro")))
              .withColumn("distancia", F.levenshtein(padronizar_texto("a.bairro"), padronizar_texto("b.bairro")))
              .filter("distancia <= 2")
              .select(F.col("a.municipio").alias("municipio"), F.col("a.bairro").alias("bairro"),
                      F.col("b.bairro").alias("parecido_com"), "distancia"))
print(f"pares parecidos: {parecidos.count()}")
display(parecidos.orderBy("distancia", "municipio"))

In [0]:
# 7e. Bairro com o nome do município (ex.: bairro RECIFE em Recife, que existe: é o Recife Antigo)
#     e bairro que aparece em vários municípios (ex.: CENTRO). Normal, mas os filtros da tela precisam do município junto
display(b.join(nomes.select(F.col("municipio").alias("bairro")).distinct(), "bairro"))
display(b.groupBy("bairro").agg(F.countDistinct("municipio").alias("municipios"), F.collect_set("municipio").alias("quais"))
         .filter("municipios > 1").orderBy(F.desc("municipios")))

In [0]:
# 7f. Série quebrada: segmento com menos de 48 meses. Pode ser UC nova ou um nome que mudou no meio da série
meses = rm.groupBy(*SEGMENTO).agg(F.count("*").alias("meses"), F.min("mes_referencia").alias("primeiro"),
                                  F.max("mes_referencia").alias("ultimo"))
print(f"segmentos com menos de 48 meses: {meses.filter('meses < 48').count()} de {meses.count()}")
display(meses.filter("meses < 48").orderBy("meses"))

In [0]:
# 8. Nomes no cadastro (Silver), que aparecem na visão 360
uc = spark.table(f"{CATALOGO}.silver.tb_unidades_consumidoras")
print("documentos (CPF/CNPJ) com mais de um nome:",
      uc.groupBy("cpf_cnpj").agg(F.countDistinct("nome_titular").alias("n")).filter("n > 1").count(), "(esperado: 0)")
print("nomes deformados:", uc.filter(F.col("nome_titular").rlike(deformado["encoding_quebrado"])
                                     | F.col("nome_titular").rlike(deformado["caractere_estranho"])
                                     | F.col("nome_titular").rlike(deformado["espaco_extra"])).count(), "(esperado: 0)")
# Homônimos (mesmo nome, documentos diferentes) são normais em nomes comuns: a busca da tela deve usar o id_uc
display(uc.groupBy("nome_titular").agg(F.countDistinct("cpf_cnpj").alias("documentos"), F.count("*").alias("ucs"))
          .filter("documentos > 1").orderBy(F.desc("documentos")).limit(500))
perfilar_coluna(uc, "nome_titular", n=10)

## O que esperar (rodada local com 100% dos dados, 21/10/2026)

| Conferência | Esperado |
|---|---|
| `agg_receita_mensal` / `agg_preco_volume` | 53.088 / 1.106 linhas; 48 meses |
| Grão, nulos e domínio | todas as checagens `ok` |
| Meses com diferença entre Gold e Silver | 0 |
| Receita de energia P1 → P2 | +8,0% (R$ 117,8 mi → R$ 127,3 mi) |
| Receita total P1 → P2 | +8,0% (R$ 157,1 mi → R$ 169,7 mi) |
| kWh P1 → P2 | −4,0% (226,7 mi → 217,7 mi) |
| Efeito-volume / efeito-preço | −R$ 4,98 mi / +R$ 14,42 mi |
| Linhas de preço × volume que não fecham | 0 |
| Nomes com mais de uma grafia / deformados | 0 / 0 |
| Pares de bairros parecidos | só bairros reais e diferentes: CAETÉS I/II/III, MARANGUAPE I/II, ARTUR LUNDGREN I/II, BONANÇA I/II, CABANGA/CAXANGÁ, MANGABEIRA/MANGUEIRA, CURADO/PRADO, TORRE/TORRÕES |
| Documento com mais de um nome | 0 (homônimos com documentos diferentes são normais) |
| Maiores quedas de volume | comercial e residencial de Iputinga e Macaxeira (Recife) e de Jardim Jordão e Prazeres (Jaboatão) |

Se algum número vier diferente, confira primeiro se as Silver de cadastro, medição e faturamento rodaram antes do `gold_receita`.